# R15-H212 RE-RUN - Clean-Conditions Recall (daily-token-cap window reset)

**Context**: the first R15-H212 attempt (`clean-recall-h212-20260708T045954Z.json`) was INCONCLUSIVE -
the Bedrock DAILY TOKEN CAP fired mid-ingest (60 "Too many tokens per day" retries -> 30 chunks lost),
replacing the wave-2 concurrency confound with a token-cap confound. Neither the CONFIRM branch nor the
DEGRADED-with-clean-telemetry branch could fire. This re-run replays the SAME v2 arm after the daily
window RESET (frontier engine smoke-tested responsive), on a freshly wiped neo4j4 scratch.

**Hypothesis**: re-running the H158 v2 arm (identity_stack v2, Bedrock Sonnet engine) under a reset daily
token budget yields failed-chunk warnings at or below the v1 arm's 76 and recall@16 >= 0.875, lifting
H158 to CONFIRMED and green-lighting the v2 default flip.

**Registered measurement** (same H158 harness, `notebooks/h158_measure.py`):
- failed-chunk warnings = `extraction.warning` events in the ingest event log (H158: v1 76, v2 119)
- pure-seed evidence recall@16 over the 24-gold H34 probe set (H158: v1 0.875, v2 0.8333)
- SAME_AS precision proxy + false-merge count vs the 298-pair H101 benchmark (continuity)

**Registered bar**: warnings <= 76 AND recall@16 >= 0.875 -> H158 CONFIRMED + default flip approved.
DEGRADED stands if recall stays low WITH clean telemetry (no token-cap casualties, failures <= 76) - in
which case per-chunk forensics run on any missing probe golds. If failures > 76 the run is INCONCLUSIVE.

Ingest was run via the CLI (init + `kgf ingest data/external/cpap-datasheets-and-manuals` on
`config-h212-v2.yml`, neo4j pinned to the fresh scratch instance `bolt://172.19.0.4:7687` per DEF-4/DEF-5),
logged to `logs/h212-rerun.log`, event log `logs/h212-v2-events.jsonl`. This notebook measures the graph.

## Imports

In [1]:
# stdlib
import json
import datetime
import re
import collections
from pathlib import Path
# third party
from neo4j import GraphDatabase
from rich import print as rprint
from rich.table import Table
from rich.console import Console
# project (CWD = project root)
import sys
sys.path.insert(0, "src")
sys.path.insert(0, ".")
from notebooks.h158_measure import precision_proxy, recall_at_k, _present, _norm
console = Console()

2026-07-08 07:38:00.256 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


## Configuration

In [2]:
FRESH_URI = "bolt://172.19.0.4:7687"
FRESH_AUTH = ("neo4j", "kgfoundry")
# The CLI APPENDS to the configured event log, so logs/h212-v2-events.jsonl carries BOTH the
# invalidated first run (ts <= 2026-07-07T23:59Z) and this re-run (ts >= 2026-07-08T05:04Z).
# Split deterministically by timestamp into a re-run-only file for the harness.
RAW_LOG = "logs/h212-v2-events.jsonl"
H212_LOG = "logs/h212-rerun-events.jsonl"
RERUN_START = "2026-07-08T05:04"
with open(H212_LOG, "w") as _out:
    for _line in Path(RAW_LOG).read_text().splitlines():
        if json.loads(_line).get("ts", "") >= RERUN_START:
            _out.write(_line + "\n")
INGEST_LOG = "logs/h212-rerun.log"
BENCH = "reports/identity-benchmark-h101-20260707-094448.json"
PROBES = "tests/probes/cpap-probe-set.yml"
H212_RECALL_OUT = "reports/h212-v2-recall-rerun.json"
TOP_K = 16
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

# H158 reference arms (the confound baseline)
H158 = {
    "v1_warnings": 76, "v2_warnings": 119,
    "v1_recall": 0.875, "v2_recall": 0.8333,
    "replay_false_merges": 11, "false_merge_tol": 0.20,
    "precision_bar": 0.50,
}
# Registered H212 bar
BAR_WARN = 76        # extraction.warning total must be <= this (v1 arm)
BAR_RECALL = 0.875   # recall@16 must be >= this (non-regression)

rprint("[bold cyan]R15-H212 RE-RUN[/bold cyan]  stamp %s  bar: warnings<=%d AND recall>=%.3f"
       % (STAMP, BAR_WARN, BAR_RECALL))
rprint("  event-log split: %d re-run lines -> %s"
       % (len(Path(H212_LOG).read_text().splitlines()), H212_LOG))

R15-H212 RE-RUN  stamp 20260708T053800Z  bar: warnings<=76 AND recall>=0.875

event-log split: 3650 re-run lines -> logs/h212-rerun-events.jsonl

## Extraction telemetry - the registered failed-chunk warning count

`extraction.warning` events in the event log are the registered failed-chunk warnings. H158 v2 carried
119 (113 dangling-relationship + 6 `chunk extraction failed` = Bedrock quota casualties under wave-2
contention); v1 carried 76 (all dangling). Under quiet Bedrock the quota-casualty class should vanish
and the total should land at or below 76.

In [3]:
warn_reasons = collections.Counter()
warn_total = 0
warn_chunks = collections.Counter()
for line in Path(H212_LOG).read_text().splitlines():
    rec = json.loads(line)
    if rec.get("event") == "extraction.warning":
        warn_total += 1
        warn_reasons[rec.get("reason", "").split(":")[0]] += 1
        warn_chunks[rec.get("chunk", "?")] += 1

ev_hist = collections.Counter()
for line in Path(H212_LOG).read_text().splitlines():
    ev_hist[json.loads(line).get("event", "?")] += 1

# token-cap retry count from the ingest log (first attempt had 60 -> 30 chunks lost)
token_cap_retries = Path(INGEST_LOG).read_text().count("Too many tokens per day")

rprint("[bold]extraction.warning total[/bold]: [yellow]%d[/yellow]  (H158 v1 %d / v2 %d)"
       % (warn_total, H158["v1_warnings"], H158["v2_warnings"]))
rprint("  by reason class: %s" % dict(warn_reasons))
chunk_fail = warn_reasons.get("chunk extraction failed", 0)
rprint("  quota-casualty class (chunk extraction failed): [yellow]%d[/yellow]  "
       "(first attempt had 30; clean Bedrock should be 0)" % chunk_fail)
rprint("  token-cap retries in ingest log: [yellow]%d[/yellow]  (first attempt had 60)"
       % token_cap_retries)
rprint("  docs=%d/%d  skipped=%d  merges=%d  defers=%d  vetoes=%d"
       % (ev_hist.get("document.completed", 0), ev_hist.get("document.started", 0),
          ev_hist.get("document.skipped", 0), ev_hist.get("resolution.merge", 0),
          ev_hist.get("resolution.defer", 0), ev_hist.get("resolution.veto", 0)))

extraction.warning total: 110  (H158 v1 76 / v2 119)

by reason class: {'dangling relationship': 60, 'chunk extraction failed': 50}

quota-casualty class (chunk extraction failed): 50  (first attempt had 30; clean Bedrock should be 0)

token-cap retries in ingest log: 100  (first attempt had 60)

docs=23/28  skipped=5  merges=347  defers=633  vetoes=49

## Graph census (fresh instance)

In [4]:
drv = GraphDatabase.driver(FRESH_URI, auth=FRESH_AUTH)
with drv.session() as s:
    ents = s.run("MATCH (e:Entity) RETURN count(e) AS c").single()["c"]
    rels = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
    chunks = s.run("MATCH (c:Chunk) RETURN count(c) AS c").single()["c"]
rprint("entities %d  relationships %d  chunks %d  (H158 v2: 2201 ents / 6808 rels)"
       % (ents, rels, chunks))

entities 1094  relationships 3946  chunks 132  (H158 v2: 2201 ents / 6808 rels)

## SAME_AS precision proxy - resolver merges vs the 298 adjudicated labels
Recomputed from the H212 event log (transitive union of `resolution.merge` events).

In [5]:
pp = precision_proxy(H212_LOG, BENCH)
rprint("[bold]precision proxy[/bold]  merges %d  defers %d  vetoes %d  | tp %d fp %d fn %d"
       % (pp["merges_total"], pp["defers_total"], pp["nli_vetoes"],
          pp["proxy_tp"], pp["proxy_fp"], pp["proxy_fn"]))
rprint("  SAME_AS precision [yellow]%.4f[/yellow] (bar >= %.2f)  false-merges [yellow]%d[/yellow] (replay 11 +-20%%)"
       % (pp["same_as_precision"], H158["precision_bar"], pp["false_merges"]))

precision proxy  merges 347  defers 633  vetoes 49  | tp 10 fp 10 fn 42

SAME_AS precision 0.5000 (bar >= 0.50)  false-merges 10 (replay 11 +-20%)

## Pure-seed evidence recall@16 (24 gold probes)
Same harness as H158; neo4j uri pointed at the fresh instance, embeddings/graphrag from base `config.yml`
(Titan, unchanged).

In [6]:
rr = recall_at_k(FRESH_URI, TOP_K, "v2-h212", PROBES)
Path(H212_RECALL_OUT).write_text(json.dumps(rr, indent=2))
rprint("[bold]recall@%d[/bold]  mean [yellow]%.4f[/yellow] (bar >= %.3f; H158 v1 %.3f / v2 %.4f)  fully-covered %d/%d"
       % (TOP_K, rr["mean_recall"], BAR_RECALL, H158["v1_recall"], H158["v2_recall"],
          rr["fully_covered"], rr["n_probes"]))
failing = {k: v for k, v in rr["per_probe"].items() if v < 1.0}
rprint("  failing/partial probes: %s" % failing)

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


...

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


2026-07-08 07:38:07.651 | INFO     | knowledge_graph_foundry.extraction.embeddings:_embed_bedrock:91 - Embeddings: 1/1
2026-07-08 07:38:07.652 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - Generated embeddings for 1/1 entities via bedrock (0 cache hits)


recall@16  mean 0.7917 (bar >= 0.875; H158 v1 0.875 / v2 0.8333)  fully-covered 17/24

failing/partial probes: {'P08': 0.0, 'P09': 0.0, 'P11': 0.5, 'P15': 0.5, 'P16': 0.0, 'P17': 0.5, 'P18': 0.5}

## Bar arithmetic and verdict

In [7]:
warn_ok = warn_total <= BAR_WARN
recall_ok = rr["mean_recall"] >= BAR_RECALL - 1e-9
# clean telemetry = failures within the v1 bar AND no token-cap casualties (first-run lesson)
clean_telemetry = warn_ok and chunk_fail == 0 and token_cap_retries == 0
fm_lo = H158["replay_false_merges"] * (1 - H158["false_merge_tol"])
fm_hi = H158["replay_false_merges"] * (1 + H158["false_merge_tol"])
fm_ok = fm_lo <= pp["false_merges"] <= fm_hi
prec_ok = pp["same_as_precision"] >= H158["precision_bar"]

if warn_ok and recall_ok:
    verdict = "CONFIRMED"
elif clean_telemetry and not recall_ok:
    verdict = "DEGRADED-stands"
else:
    verdict = "INCONCLUSIVE"

tbl = Table(title="R15-H212 RE-RUN - clean-conditions vs H158 arms")
for h in ["metric", "H158 v1", "H158 v2", "H212 rerun", "bar", "pass"]:
    tbl.add_column(h)
tbl.add_row("extraction.warning total", str(H158["v1_warnings"]), str(H158["v2_warnings"]),
            str(warn_total), "<= %d" % BAR_WARN, "PASS" if warn_ok else "FAIL")
tbl.add_row("token-cap retries", "0", "-", str(token_cap_retries), "= 0",
            "PASS" if token_cap_retries == 0 else "FAIL")
tbl.add_row("recall@16 (24 probes)", "%.4f" % H158["v1_recall"], "%.4f" % H158["v2_recall"],
            "%.4f" % rr["mean_recall"], ">= %.3f" % BAR_RECALL, "PASS" if recall_ok else "FAIL")
tbl.add_row("SAME_AS precision proxy", "-", "0.5217", "%.4f" % pp["same_as_precision"],
            ">= %.2f" % H158["precision_bar"], "PASS" if prec_ok else "FAIL")
tbl.add_row("false merges", "-", "11", str(pp["false_merges"]),
            "%.0f-%.0f" % (fm_lo, fm_hi), "PASS" if fm_ok else "FAIL")
console.print(tbl)
rprint("[bold]R15-H212 RE-RUN VERDICT: %s[/bold]" % verdict)

                R15-H212 RE-RUN - clean-conditions vs H158 arms                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━┓
┃ metric                   ┃ H158 v1 ┃ H158 v2 ┃ H212 rerun ┃ bar      ┃ pass ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━┩
│ extraction.warning total │ 76      │ 119     │ 110        │ <= 76    │ FAIL │
│ token-cap retries        │ 0       │ -       │ 100        │ = 0      │ FAIL │
│ recall@16 (24 probes)    │ 0.8750  │ 0.8333  │ 0.7917     │ >= 0.875 │ FAIL │
│ SAME_AS precision proxy  │ -       │ 0.5217  │ 0.5000     │ >= 0.50  │ PASS │
│ false merges             │ -       │ 11      │ 10         │ 9-13     │ PASS │
└──────────────────────────┴─────────┴─────────┴────────────┴──────────┴──────┘

R15-H212 RE-RUN VERDICT: INCONCLUSIVE

## Per-chunk forensics on missing probe golds

Registered branch: if recall stays low with clean telemetry, adjudicate every missing gold across three
levels - is the gold string in the source chunks (readable document text), in the graph (extraction +
load captured it as an entity/property), and in the probe's top-16 seed render (retrieval reachability).
A gold present in chunks but absent from the graph is extraction loss; present in the graph but not in
seeds is a retrieval/identity miss. Runs only when at least one probe is below 1.0.

In [8]:
import yaml
probes = [p for p in yaml.safe_load(Path(PROBES).read_text()) if p.get("gold_evidence")]
probe_by_id = {p["id"]: p for p in probes}

def gold_in_chunks(session, gold):
    for r in session.run("MATCH (c:Chunk) RETURN c.text AS t"):
        if r["t"] and _present(gold, _norm(r["t"])):
            return True
    return False

def gold_in_graph(session, gold):
    for r in session.run("MATCH (e:Entity) RETURN e.name AS name, e.description AS d, properties(e) AS props"):
        spec = {k.removeprefix("prop_"): v for k, v in (r["props"] or {}).items() if k.startswith("prop_")}
        blob = _norm(" ".join([str(r["name"] or ""), str(r["d"] or ""), json.dumps(spec, default=str)]))
        if _present(gold, blob):
            return True
    return False

forensics = {}
if failing:
    from copy import deepcopy
    from knowledge_graph_foundry import load_settings
    from knowledge_graph_foundry.extraction import generate_embeddings
    from knowledge_graph_foundry.graph.graphrag import vector_query
    from knowledge_graph_foundry.models import Entity
    from knowledge_graph_foundry.pipeline import Foundry
    from notebooks.h158_measure import _render_nodes

    base = load_settings(Path("config.yml"))
    vec = base.graphrag.vector_index_name
    st = deepcopy(base)
    st.neo4j.uri, st.neo4j.user, st.neo4j.password = FRESH_URI, "neo4j", "kgfoundry"
    st.graphrag.top_k = TOP_K
    with Foundry(st) as f:
        for pid in failing:
            p = probe_by_id[pid]
            q = p["question"]
            pe = Entity.create(q[:80], types=["Query"], description=q)
            emb = generate_embeddings([pe], st.embeddings)[0].embedding
            seeds = [srow["id"] for srow in vector_query(f.driver, emb, vec, top_k=TOP_K)]
            with f.driver.session() as sess:
                ctx = _render_nodes(sess, seeds)
                per_gold = []
                for g in p["gold_evidence"]:
                    per_gold.append({
                        "gold": g,
                        "in_source_chunks": bool(gold_in_chunks(sess, g)),
                        "in_graph": bool(gold_in_graph(sess, g)),
                        "in_top16_seeds": bool(_present(g, ctx)),
                    })
            forensics[pid] = {"question": q, "sources": p.get("sources"),
                              "score": failing[pid], "golds": per_gold}
    for pid, fx in forensics.items():
        rprint("[bold]%s[/bold] (%.2f)  %s" % (pid, fx["score"], fx["question"][:70]))
        for g in fx["golds"]:
            ok = g["in_source_chunks"] and g["in_graph"] and g["in_top16_seeds"]
            flag = "" if ok else "  <-- MISS"
            rprint("    gold '%s'  chunk=%s graph=%s seeds=%s%s"
                   % (g["gold"], g["in_source_chunks"], g["in_graph"], g["in_top16_seeds"], flag))
else:
    rprint("[green]no failing probes - forensics not required[/green]")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=50, offset=49>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 49, 'line': 1, 'column': 50}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id:$id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


...

gold '0-60 mins'  chunk=True graph=True seeds=False  <-- MISS

gold '0 to 45 min'  chunk=True graph=True seeds=True

P18 (0.50)  Which is quieter, the ResMed AirStart 10 or the ResMed AirSense 11?

gold '26.6 dBA'  chunk=True graph=False seeds=False  <-- MISS

gold '27 dBA'  chunk=True graph=True seeds=True

## Write report

In [9]:
report = {
    "hypothesis": "R15-H212", "rerun": True, "stamp": STAMP, "verdict": verdict,
    "fresh_instance": FRESH_URI, "config": "config-h212-v2.yml",
    "ingest_log": INGEST_LOG, "event_log": H212_LOG, "raw_event_log": RAW_LOG,
    "prior_attempt": "reports/clean-recall-h212-20260708T045954Z.json",
    "bar": {"warnings_le": BAR_WARN, "recall_ge": BAR_RECALL},
    "h158_reference": H158,
    "telemetry": {
        "extraction_warning_total": warn_total,
        "warning_by_reason": dict(warn_reasons),
        "chunk_extraction_failed": chunk_fail,
        "token_cap_retries": token_cap_retries,
        "event_histogram": dict(ev_hist),
    },
    "census": {"entities": ents, "relationships": rels, "chunks": chunks},
    "precision_proxy": pp,
    "recall": rr,
    "clauses": {
        "warnings_le_76": bool(warn_ok),
        "token_cap_zero": bool(token_cap_retries == 0),
        "recall_ge_0875": bool(recall_ok),
        "precision_ge_50pct": bool(prec_ok),
        "false_merges_within_tol": bool(fm_ok),
    },
    "forensics": forensics,
    "deviations": [
        "event log logs/h212-v2-events.jsonl is append-mode across runs; harness reads the "
        "timestamp-split re-run-only file logs/h212-rerun-events.jsonl (split at 2026-07-08T05:04)",
    ],
}
out = Path("reports/clean-recall-h212-rerun-%s.json" % STAMP)
out.write_text(json.dumps(report, indent=2, default=str))
rprint("[bold]wrote[/bold] %s" % out)

wrote reports/clean-recall-h212-rerun-20260708T053800Z.json